In [ ]:
import pandas as pd

data = pd.read_csv('../data/herg_dataset.csv')
data.head()

In [ ]:
data = data[['SMILES', 'PIC50']]
data.rename(columns={'PIC50': 'pic50', 'SMILES': 'smiles'}, inplace=True)
data.head()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
sns.histplot(data, x='pic50', bins=30, kde=True)
plt.show()

In [ ]:
print(len(data))
data = data[data['pic50'] > 2].reset_index(drop=True)
len(data)

In [ ]:
len(data)

In [ ]:
from rdkit import Chem

mols = [Chem.MolFromSmiles(smi) for smi in data['smiles']]
valid_mols_ids = [i for i, m in enumerate(mols) if m is not None]
data = data.iloc[valid_mols_ids].reset_index(drop=True)
len(data)

In [ ]:
mols = [m for m in mols if m is not None]
len(mols)

In [ ]:
from src.core.utils import select_diverse_subset_butina

num_samples = 200
diverse_indices = select_diverse_subset_butina(mols, num_samples, similarity_cutoff=0.6)
data = data.iloc[diverse_indices].reset_index(drop=True)
data.head()

In [ ]:
len(data)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
sns.histplot(data, x='pic50', bins=30, kde=True)
plt.show()

In [ ]:
from src.core.fingerprints import Fingerprints
df = data

names = ['ecfp']
params = {
    'ecfp': {'radius': 2, 'size': 1024, 'count': True},
}
fingerprints, f_names = Fingerprints().apply(
    smiles=df['smiles'].tolist(),
    names=names,
    **params
)
df_fingerprints = pd.DataFrame(fingerprints, columns=f_names)
threshold = 0.85
df_fingerprints = df_fingerprints[[col for col in df_fingerprints.columns if df_fingerprints[col].value_counts(normalize=True).iloc[0] <= threshold]]
df_fingerprints = df_fingerprints.loc[:, df_fingerprints.nunique() > 1]
df_fingerprints['pic50'] = df['pic50'].values
df_fingerprints['smiles'] = df['smiles'].values

df_fingerprints

In [ ]:
df_fingerprints.to_csv('../data/real_data/data_herg_ecfp.csv', index=False)